# Extract per-residue MINT embeddings on Colab

Runs `interaction_head/extract_residue_embeddings.py` on a Colab GPU.

1. **Runtime → Change runtime type → GPU** (T4 is enough; the model runs in fp16).
2. Put your input JSONL (e.g. `trainset.json` from `interface/label_interfaces.py`) in Google Drive and set `INPUT` below.
3. Shards are written straight to Drive. The script skips shards that already exist, so if the
   session disconnects, just re-run all cells and it picks up where it left off.

In [1]:
# ---- settings ----
DRIVE_DIR  = "/content/drive/MyDrive/mint"           # working folder on Drive
INPUT      = f"{DRIVE_DIR}/trainset_contacts.json"            # JSONL from interface/label_interfaces.py
OUT_DIR    = f"{DRIVE_DIR}/emb_train"                # shard_XXXXX.pt files go here
CKPT       = f"{DRIVE_DIR}/mint.ckpt"                # cached here so it's only downloaded once (~3.2 GB)

REPO_URL   = "https://github.com/AlecSilver/mint.git"
BRANCH     = "interaction_head"

MAX_TOKENS = 16384   # token budget per forward pass; 4096 is the script default (sized for a 6 GB GPU)
CROP       = "random"  # random | interface | center | start
LAYERS     = "30"      # MINT layer(s) to save, e.g. "24 28 33"; each extra layer adds another full copy of the embeddings
EXTRA_ARGS = ""      # e.g. "--labeled-only" or "--limit 200"

In [2]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.makedirs(DRIVE_DIR, exist_ok=True)
assert os.path.exists(INPUT), f"upload your input JSONL to {INPUT}"

Mounted at /content/drive


In [3]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


## Get the code and checkpoint

If the repo is private, use a URL with a personal access token instead
(`https://<token>@github.com/AlecSilver/mint.git`), or upload the repo to Drive and point `REPO_DIR` at it.

In [4]:
REPO_DIR = "/content/mint"
if not os.path.exists(REPO_DIR):
    !git clone -q --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q

# Colab already has torch, pandas and tqdm, which is all the script needs
!pip install -q --no-deps -e {REPO_DIR}

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for mint (pyproject.toml) ... done


In [5]:
if not os.path.exists(CKPT):
    !wget -q --show-progress -O {CKPT}.part https://huggingface.co/varunullanat2012/mint/resolve/main/mint.ckpt
    os.replace(CKPT + ".part", CKPT)
!ls -lh {CKPT}

-rw------- 1 root root 3.1G Oct 27 16:43 /content/drive/MyDrive/mint/mint.ckpt


## Quick test

Embeds the first 20 pairs into a scratch folder to check that everything works.

In [6]:
%cd {REPO_DIR}/interaction_head
!rm -rf /content/emb_test
!python extract_residue_embeddings.py {INPUT} /content/emb_test --checkpoint {CKPT} --max-tokens {MAX_TOKENS} --crop {CROP} --layers {LAYERS} --limit 20

/content/mint/interaction_head
100% 20/20 [00:07<00:00,  2.55pair/s]


In [7]:
import torch
shard = torch.load("/content/emb_test/shard_00000.pt")
for k, v in shard.items():
    print(f"{k:12s}", tuple(v.shape) if hasattr(v, "shape") else f"list[{len(v)}]", getattr(v, "dtype", ""))
print("layers:", shard["layers"])
print("interface residues:", (shard["labels"] == 1).sum().item(), "/", (shard["labels"] >= 0).sum().item(), "labelled")

ids          list[20] 
layers       list[1] 
cluster      (20,) torch.int64
emb          (8844, 1, 1280) torch.float16
labels       (8844,) torch.int8
chain        (8844,) torch.int8
resid        (8844,) torch.int32
offsets      (21,) torch.int64
crop_start   (20, 2) torch.int32
full_len     (20, 2) torch.int32
layers: [30]
interface residues: 1990 / 7694 labelled


## Full run

Safe to re-run after a disconnect: finished shards are skipped. If you hit CUDA out-of-memory, lower `MAX_TOKENS`.

In [8]:
%cd {REPO_DIR}/interaction_head
!python extract_residue_embeddings.py {INPUT} {OUT_DIR} --checkpoint {CKPT} --max-tokens {MAX_TOKENS} --crop {CROP} --layers {LAYERS} {EXTRA_ARGS}

/content/mint/interaction_head
100% 45653/45653 [04:01<00:00, 189.28pair/s]


In [9]:
!ls {OUT_DIR} | wc -l
!du -sh {OUT_DIR}

230
61G	/content/drive/MyDrive/mint/emb_train
